In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import xarray as xr
import numpy as np
import os
# import pandas as pd
from glob import glob
#import climpredNEW.climpred 
#from climpredNEW.climpred.options import OPTIONS
from climpred.options import OPTIONS
import climpred
import pickle
from mpl_toolkits.basemap import Basemap
from numpy import meshgrid
from mpl_toolkits.axes_grid1.axes_divider import make_axes_locatable
import matplotlib.colors as mcolors
import cartopy.feature as cfeature
import itertools
import cartopy.crs as ccrs
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter, LatitudeLocator
import matplotlib.ticker as mticker
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, TwoSlopeNorm
from scipy.stats import rankdata
import bottleneck as bn
import scipy.stats as stats

from function import preprocessUtils as putils
from function import masks
from function import verifications
from function import funs as f
from function import conf
from function import loadbias
from function import dataLoad
from function import quikplot as qp


/glade/work/klesinger/conda-envs/tf212gpu_new/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Save ACC values for the later plots. 


In [2]:
region_name = 'CONUS' #or ['australia', 'CONUS', 'china']
obs_source = 'GLEAM' #['ERA5','GLEAM']

if obs_source == 'ERA5':
    soil_dir = conf.era_data
elif obs_source == 'GLEAM':
    soil_dir = conf.gleam_data



In [3]:
global obs_original,obs_raw
obs_original,obs_raw = dataLoad.load_rzsm_observations(soil_dir, region_name)
obs_original["time"] = obs_original["time"].dt.floor("D")
obs_raw["time"] = obs_raw["time"].dt.floor("D")


if region_name == 'CONUS' and obs_source=='ERA5':
    obs_original = obs_original.rename({'X':'longitude','Y':'latitude'})


obs_anom_climp = verifications.rename_obs_for_climpred(obs_original)

mask, mask_anom = masks.load_mask_vals(region_name)

In [4]:


start_obs = '2000-01-01' #Beginning of observation period for analysis. We actually have data starting from 1999 so that we could have a 7-day rolling mean applied to the data and have up to 12 weeks lags for RZSM
end_obs = '2020-12-31' #end of observations for ERA5 and GLEAM. We actually needed data through 2020-02-15 since we have an initialization on 2019-12-25
start_testing = '2018-01-01' #Beginning of testing period
end_testing = '2019-12-31'
train_end_string = '2015-12-31' #last string date for training
train_end = 2015 #last year of training dates


global RZSM_or_Tmax_or_both
RZSM_or_Tmax_or_both = 'RZSM' # for getting the predictor from either RZSM and Tmax ('both') or only RZSM ('RZSM')

lead_select = [6,13,20,27]

In [5]:
def print_min_max(file,name):
    print(name)
    print(f'Maximum value in file is {np.nanmax(file[putils.xarray_varname(file)])}')
    print(f'Minimum value in file is {np.nanmin(file[putils.xarray_varname(file)])}')

In [6]:
global verification_var
verification_var = f'soilw_bgrnd_{obs_source}' #this is for what we are verifying with DL outputs

#Gleam observations
gleam_dir = f'{soil_dir}/{region_name}'

#Forecast predictions
gefsv12_fcst_dir = f'{conf.gefsv12_data}/{region_name}'

#ERA5 observations
era5_dir = f'{conf.era_data}/{region_name}'


In [7]:
#Load observation anomaly
gleam_anom = verifications.load_RZSM_anomaly_obs(region_name, soil_dir).load()
ecmwf_anom = verifications.load_ECMWF_baseline_anomaly(region_name).load()
gefs_anom = verifications.load_GEFSv12_baseline_anomaly(region_name).load()

'''Load a base file to serve as the xarray template to add our predictions from UNET into.'''
base_file_testing = gefs_anom.copy(deep=True).sel(S=slice(start_testing, None)).load()

global lat, lon
lat = base_file_testing.Y.values  # for plotting later
lon = base_file_testing.X.values


Loading /glade/derecho/scratch/klesinger/FD_RZSM_deep_learning/Data/reanalysis/GLEAM baseline anomaly files
Loading ECMWF baseline anomaly files
Loading GEFSv12 baseline anomaly files


In [8]:
# #Load the percentile files from 
# ecmwf_perc = verifications.load_ECMWF_percentile_anomaly(region_name).load()
# gefs_perc = verifications.load_GEFSv12_percentile_anomaly(region_name).load()

In [9]:
#bias corrected data
gef_bc_crps, ecm_bc_crps = loadbias.load_additive_bias_corrected_data_CRPS(lead_select,region_name,obs_source)


# Now create baseline anomalies of files to not have to re-compute later

In [10]:
#First
#Create seasonal anomaly from observations
print(f'Creating the seasonal anomalies for observational data and then subsetting for everything after {start_testing} date.')
save_anomaly_dir = f'Data/anomaly/{region_name}'
os.system(f'mkdir -p {save_anomaly_dir}')

obs_RZSM_save = f'{save_anomaly_dir}/obs_RZSM_anomaly_testing_{obs_source}.nc'
ref_RZSM_save = f'{save_anomaly_dir}/reforecast_RZSM_anomaly_testing_{obs_source}.nc'


Creating the seasonal anomalies for observational data and then subsetting for everything after 2018-01-01 date.


In [11]:
def open_file_create_seasonal_anomaly(path,train_end):
    #Must subset by lead first (because we actually have data previously from past lag weeks)
    return(create_seasonal_anomaly(xr.open_mfdataset(path).sel(L=slice(0,34)).rolling(L=7, min_periods=7,center=False).mean(),train_end=train_end,source='reforecast'))

def check_values_in_file(file,lead):
    '''Just print some values to verify the files aren't identical when comparing with other results'''
    name_file = list(file.keys())[0]
    return(print(file[name_file].isel(L=lead).isel(M=10).isel(S=0).values))
    

In [12]:

def load_experiment_predictions_and_observations(lead,experiment,region_name,obs_source):
    # #Test
    # experiment='EX0'
    day_num = (lead*7)-1
    min_max_dir = f'Data/min_max_values/{region_name}'
    verification_directory = f'Data/model_npy_input_data/{region_name}/Verification_data' #For observation verification
    # bias_correction_dir = f'Data/bias_mean_values/Wk_{lead}'

    ex_name = experiment


    #Load the actual observations (used for the Mean Absolute Error calculation)
    obs_final_train,obs_final_validation,obs_final_testing = f.load_verification_observations_updated(lead,verification_directory,obs_source)
    obs_RZSM = np.array(obs_final_testing) #anomaly

    #Convert observations 0 values to nan (only for the RZSM observations). These values had a zero where there is no land soil moisture
    obs_RZSM = np.where(obs_RZSM == 0,np.nan,obs_RZSM)
    
    obs_RZSM =verifications.reverse_min_max_scaling(obs_RZSM,region_name,day_num,'GEFSv12',2019)

    predictions_directory = f'predictions/{region_name}/Wk{lead}_testing'

    cont=False
    ecmwf_present=False
    if obs_source == 'GLEAM':
        prediction_GEFS = np.load(f'{predictions_directory}/Wk{lead}_testing_{ex_name}_regular_RZSM.npy')
        try:
            prediction_ECMWF = np.load(f'{predictions_directory}/Wk{lead}_testing_{ex_name}_ECMWF_regular_RZSM.npy')
            ecmwf_present=True
        except FileNotFoundError:
            prediction_ECMWF = np.empty_like(prediction_GEFS)
            
        cont = True
    elif obs_source == 'ERA5':
        prediction_GEFS = np.load(f'{predictions_directory}/Wk{lead}_testing_{ex_name}_regular_ERA5_RZSM.npy')
        try:
            prediction_ECMWF = np.load(f'{predictions_directory}/Wk{lead}_testing_{ex_name}_ECMWF_regular_ERA5_RZSM.npy')
            ecmwf_present=True
        except:
            prediction_ECMWF = np.empty_like(prediction_GEFS)
            
        cont = True


    if cont:
        print(f'Test prediction shape: {prediction_GEFS.shape}')
        prediction_RZSM_GEFS = verifications.reverse_min_max_scaling(prediction_GEFS[-1,:,:,:],region_name, day_num, 'GEFSv12',2019)
        if ecmwf_present:
            prediction_RZSM_ECMWF = verifications.reverse_min_max_scaling(prediction_ECMWF[-1,:,:,:],region_name, day_num, 'GEFSv12',2019)
        else:
            prediction_RZSM_ECMWF = np.zeros_like(prediction_RZSM_GEFS)
            prediction_RZSM_ECMWF[:,:,:,:] = np.nan
        print(f'Shape of prediction RZSM: {prediction_RZSM_GEFS.shape}')
    
        #Convert back to np.nan values for the ocean and other water bodies
        prediction_RZSM_GEFS = np.where(np.isnan(obs_RZSM),np.nan,prediction_RZSM_GEFS.squeeze())
        if ecmwf_present:
            prediction_RZSM_ECMWF = np.where(np.isnan(obs_RZSM),np.nan,prediction_RZSM_ECMWF.squeeze())
    
        return(prediction_RZSM_GEFS, prediction_RZSM_ECMWF, obs_RZSM)
    else:
        return(np.zeros(shape=obs_RZSM.shape),np.zeros(shape=obs_RZSM.shape), obs_RZSM)


In [13]:
def return_non_post_processed_forecasts(lead,dim_order):
    '''We are selecting a single lead time, so use this code'''
    if lead == 0:
        index_sel = 0
    else:
        index_sel = (lead*7)-1
    

    if region_name == 'CONUS':
        RZSM_base_reforecast_climpred_GEF = f.restrict_to_CONUS_bounding_box(gefs_anom,mask).sel(L=(lead*7)-1).expand_dims({'L': 1}).transpose(*dim_order)
        RZSM_base_reforecast_climpred_ECM = f.restrict_to_CONUS_bounding_box(ecmwf_anom,mask).sel(L=(lead*7)-1).expand_dims({'L': 1}).transpose(*dim_order)
    else:
        RZSM_base_reforecast_climpred_GEF = gefs_anom.sel(L=(lead*7)-1).expand_dims({'L': 1}).transpose(*dim_order)
        RZSM_base_reforecast_climpred_ECM = ecmwf_anom.sel(L=(lead*7)-1).expand_dims({'L': 1}).transpose(*dim_order)
        
    print_min_max(RZSM_base_reforecast_climpred_GEF,'GEFS RZSM baseline value from reforecast (training, validation, testing) (no pre-processing other than anomaly computed.)')
    print_min_max(RZSM_base_reforecast_climpred_ECM,'ECMWF RZSM baseline value from reforecast (training, validation, testing) (no pre-processing other than anomaly computed.)')

    if RZSM_or_Tmax_or_both == 'both':
        return(RZSM_base_reforecast_climpred_GEF, RZSM_base_reforecast_climpred_ECM,tmax_base_reforecast_climpred)
    else:
        return(RZSM_base_reforecast_climpred_GEF, RZSM_base_reforecast_climpred_ECM)

    '''Then load the actual UNET predictions and keep the observations that are already in a good format for comparison
    This data has been loaded, converted back to anomalies, and masked for RZSM for non-land regions'''
    
    

In [14]:
def convert_prediction_to_SubX_format(file,lead,dim_order):

    if region_name == 'CONUS':
        cp_base = f.restrict_to_CONUS_bounding_box(base_file_testing.copy(deep=True).sel(L=(lead*7)-1),mask).expand_dims({'L': 1})
    else:
        cp_base = base_file_testing.copy(deep=True).sel(L=(lead*7)-1).expand_dims({'L': 1})

    
    #reshape data back to original format (testing shape)
    
    cp_base = cp_base.transpose(*dim_order)
    
    #Reshape prediction file
    file = file.reshape((104,11,1,48,96))
    
    var_OUT = xr.Dataset(
            data_vars = dict(
                file_name = (['S','M','L','Y','X'],  file[:,:,:,:,:]),
            ),
            coords = dict(
                S = cp_base.S.values,
                X = cp_base.X.values,
                Y = cp_base.Y.values,
                L = cp_base.L.values,
                M = cp_base.M.values,

            ),
            attrs = dict(
                Description = 'New data added to file'),
        )  

    return(var_OUT)


In [15]:
def add_attrs_and_lead_day(file, name_of_lead):
    # obs_clim['L'] = np.ceil(obs_clim['L'].values/7)
    xr.set_options(keep_attrs=True)
    file[name_of_lead].attrs['units'] = 'days'
    file[name_of_lead].attrs
    
    '''Must change the day to ensure climpred computes correctly and changes attrs to weeks'''
    file[name_of_lead]=file[name_of_lead]+1
    return file

In [16]:
def crps_(lead,experiment,ACC_dictionary,obs_anom_climp,gef_bc_acc,ecm_bc_acc,obs_source):
    output_dictionary = {}
    out_name = ''
    
    #Must re-add back L as a date
    dim_order = ['S','M','L','Y','X']

    RZSM_base_reforecast_climpred_GEF, RZSM_base_reforecast_climpred_ECM = return_non_post_processed_forecasts(lead,dim_order) #Returns the original reforecasts
    
    prediction_RZSM_GEFS, prediction_RZSM_ECMWF, obs_RZSM = load_experiment_predictions_and_observations(lead,experiment,region_name,obs_source) #Returns the UNET prediction and observations


    #Change name for climpred processing
    #Reforecast prediction
    prediction_RZSM_climpred_GEF = verifications.rename_subx_for_climpred(convert_prediction_to_SubX_format(file=prediction_RZSM_GEFS,lead=lead,dim_order = dim_order))
    prediction_RZSM_climpred_ECM = verifications.rename_subx_for_climpred(convert_prediction_to_SubX_format(file=prediction_RZSM_ECMWF,lead=lead,dim_order = dim_order))

    prediction_RZSM_climpred_GEF  = prediction_RZSM_climpred_GEF.rename({'file_name':'RZSM'})
    prediction_RZSM_climpred_ECM  = prediction_RZSM_climpred_ECM.rename({'file_name':'RZSM'})
    
    print_min_max(prediction_RZSM_climpred_ECM,'ECMWF RZSM anomaly prediction value from UNET')
    print_min_max(prediction_RZSM_climpred_GEF,'GEFSv12 RZSM anomaly prediction value from UNET')

    prediction_RZSM_GEFS, prediction_RZSM_ECMWF = add_attrs_and_lead_day(prediction_RZSM_climpred_GEF,'lead'), add_attrs_and_lead_day(prediction_RZSM_climpred_ECM,'lead')
    prediction_RZSM_GEFS['lead'].attrs
    
    unet_acc_gef = verifications.create_climpred_CRPS(prediction_RZSM_climpred_GEF, obs_anom_climp)
    unet_acc_ecm = verifications.create_climpred_CRPS(prediction_RZSM_climpred_ECM, obs_anom_climp)
    unet_acc_gef['lead']
    unet_acc_gef['lead'].attrs
    ACC_dictionary[f'Wk{lead}_{experiment}_MEM_RZSM_CRPS'] = unet_acc_gef
    ACC_dictionary[f'Wk{lead}_{experiment}_ECMWF_MEM_RZSM_CRPS'] = unet_acc_ecm

    #Base reforecast (before post-processing)
    base_RZSM_climpred_GEFS  = verifications.rename_subx_for_climpred(RZSM_base_reforecast_climpred_GEF).sel(init=slice(start_testing, None))
    print_min_max(base_RZSM_climpred_GEFS,'GEFS RZSM baseline anomaly from reforecast. No post-processing. \n')

    base_RZSM_climpred_ECMWF  = verifications.rename_subx_for_climpred(RZSM_base_reforecast_climpred_ECM).sel(init=slice(start_testing, None))
    print_min_max(base_RZSM_climpred_ECMWF,'ECMWF RZSM baseline anomaly from reforecast. No post-processing. \n')

    base_RZSM_climpred_GEFS = add_attrs_and_lead_day(base_RZSM_climpred_GEFS,'lead')
    base_RZSM_climpred_ECMWF = add_attrs_and_lead_day(base_RZSM_climpred_ECMWF,'lead')
    
    gefs_acc = verifications.create_climpred_CRPS_no_chunk(base_RZSM_climpred_GEFS, obs_anom_climp)
    ecm_acc = verifications.create_climpred_CRPS_no_chunk(base_RZSM_climpred_ECMWF, obs_anom_climp)
    

    ACC_dictionary[f'Wk{lead}_GEFS_MEM_baseline_RZSM_CRPS'] = gefs_acc
    ACC_dictionary[f'Wk{lead}_ECMWF_MEM_baseline_RZSM_CRPS'] = ecm_acc

    '''Do not include the bias-corrected outputs'''
    # ACC_dictionary[f'Wk{lead}_GEFS_MEM_BC_baseline_RZSM_CRPS'] = gef_bc_acc[putils.xarray_varname(gef_bc_acc)][lead-1,:,:]
    # ACC_dictionary[f'Wk{lead}_ECMWF_MEM_BC_baseline_RZSM_CRPS'] = ecm_bc_acc[putils.xarray_varname(ecm_bc_acc)][lead-1,:,:]
    

    return(ACC_dictionary)
    


In [17]:
def crps_by_season(lead, experiment, ACC_dictionary, obs_anom_climp, gef_bc_acc, ecm_bc_acc, obs_source):
    output_dictionary = {}
    out_name = ''
    
    # Must re-add back L as a date
    dim_order = ['S','M','L','Y','X']

    # Get the original reforecasts
    RZSM_base_reforecast_climpred_GEF, RZSM_base_reforecast_climpred_ECM = return_non_post_processed_forecasts(lead, dim_order)

    RZSM_base_reforecast_climpred_GEF, RZSM_base_reforecast_climpred_ECM = add_attrs_and_lead_day(RZSM_base_reforecast_climpred_GEF,'L'), add_attrs_and_lead_day(RZSM_base_reforecast_climpred_ECM,'L')
    
    # Get the UNET predictions and observations
    prediction_RZSM_GEFS, prediction_RZSM_ECMWF, obs_RZSM = load_experiment_predictions_and_observations(lead, experiment, region_name, obs_source)

    # Convert predictions to climpred format
    prediction_RZSM_climpred_GEF = add_attrs_and_lead_day(verifications.rename_subx_for_climpred(
        convert_prediction_to_SubX_format(file=prediction_RZSM_GEFS, lead=lead, dim_order=dim_order)
    ).rename({'file_name':'RZSM'}),'lead')

    prediction_RZSM_climpred_ECM = add_attrs_and_lead_day(verifications.rename_subx_for_climpred(
        convert_prediction_to_SubX_format(file=prediction_RZSM_ECMWF, lead=lead, dim_order=dim_order)
    ).rename({'file_name':'RZSM'}),'lead')
    
    # Print stats
    print_min_max(prediction_RZSM_climpred_ECM, 'ECMWF RZSM anomaly prediction value from UNET')
    print_min_max(prediction_RZSM_climpred_GEF, 'GEFSv12 RZSM anomaly prediction value from UNET')

    def add_single_month(months, season):
        vals = months[season]
        #This is to ensure that we have all of the data correctly forecasted within the distribution
        next_number = vals[-1] + 1  # Get the last number and add 1
        vals.append(next_number)
        return vals
        
    # Function to filter data by season
    def filter_by_season(data, season, obs_fcst):
        months = {
            'DJF': [12, 1, 2],
            'MAM': [3, 4, 5],
            'JJA': [6, 7, 8],
            'SON': [9, 10, 11]
        }
        if obs_fcst=='forecast':
            return data.sel(init=data['init'].dt.month.isin(months[season]))
        else:
            return data.sel(time=data['time'].dt.month.isin(months[season]))

    

    seasons = ['DJF', 'MAM', 'JJA', 'SON']

    for season in seasons:
        # Filter by season
        pred_season_GEF = filter_by_season(prediction_RZSM_climpred_GEF, season, 'forecast')
        pred_season_ECM = filter_by_season(prediction_RZSM_climpred_ECM, season, 'forecast')
        obs_season = filter_by_season(obs_anom_climp, season, 'obs')
        
        # Compute ACC for each season
        unet_acc_gef = verifications.create_climpred_CRPS_no_chunk(pred_season_GEF, obs_season)
        unet_acc_ecm = verifications.create_climpred_CRPS_no_chunk(pred_season_ECM, obs_season)
        
        ACC_dictionary[f'{season}_Wk{lead}_{experiment}_MEM_RZSM_CRPS'] = unet_acc_gef
        ACC_dictionary[f'{season}_Wk{lead}_{experiment}_ECMWF_MEM_RZSM_CRPS'] =unet_acc_ecm
        
        # Baseline reforecasts
        base_RZSM_climpred_GEFS = verifications.rename_subx_for_climpred(RZSM_base_reforecast_climpred_GEF).sel(init=slice(start_testing, None))
        base_RZSM_climpred_ECMWF = verifications.rename_subx_for_climpred(RZSM_base_reforecast_climpred_ECM).sel(init=slice(start_testing, None))
        
        # Filter baseline forecasts by season
        base_season_GEF = filter_by_season(base_RZSM_climpred_GEFS, season, 'forecast')
        base_season_ECM = filter_by_season(base_RZSM_climpred_ECMWF, season, 'forecast')
        
        gefs_acc = verifications.create_climpred_CRPS_no_chunk(base_season_GEF, obs_season)
        ecm_acc = verifications.create_climpred_CRPS_no_chunk(base_season_ECM, obs_season)
        gefs_acc['lead']
        gefs_acc['lead'].attrs
        
        # Store baseline ACC values for each season
        ACC_dictionary[f'{season}_Wk{lead}_GEFS_MEM_baseline_RZSM_CRPS'] = gefs_acc
        ACC_dictionary[f'{season}_Wk{lead}_ECMWF_MEM_baseline_RZSM_CRPS'] =ecm_acc
        
        # Bias-corrected ACC values for each season
        # ACC_dictionary[f'{season}_Wk{lead}_GEFS_MEM_BC_baseline_RZSM_CRPS'] = gef_bc_acc[putils.xarray_varname(gef_bc_acc)][lead-1,:,:]
        # ACC_dictionary[f'{season}_Wk{lead}_ECMWF_MEM_BC_baseline_RZSM_CRPS'] =ecm_bc_acc[putils.xarray_varname(ecm_bc_acc)][lead-1,:,:]

    return ACC_dictionary


In [18]:
def run_CRPS(lead,region_name,obs_source):
    print(f'Working on lead {lead}')
    # lead=1

    # save_dict_dir = f'Outputs/crps_mae/{region_name}/Wk_{lead}'
    # os.system(f'mkdir -p {save_dict_dir}')
    
    if lead <=4:
        if region_name == 'CONUS':
            if obs_source == 'GLEAM':
                
                experiment_list = [f'EX{i}' for i in range(0,30)]
                experiment_list.remove('EX26')
                if lead <=2:
                    experiment_list.remove('EX18')
                    experiment_list.remove('EX19')
                    experiment_list.remove('EX20')
                    experiment_list.remove('EX21')
            elif obs_source == 'ERA5':
                experiment_list = ['EX29']
        else:
            experiment_list = ['EX29']
    elif lead ==5:
        experiment_list = ['EX26']

    
    ACC_dictionary = {}
    ACC_season = {}
    for experiment in experiment_list:
        ACC_dictionary.update(crps_(lead=lead,experiment=experiment,
                                                              ACC_dictionary=ACC_dictionary, 
                                                              obs_anom_climp=obs_anom_climp,
                                                              gef_bc_acc=gef_bc_crps, ecm_bc_acc=ecm_bc_crps,
                                                     obs_source=obs_source))
        ACC_season.update(crps_by_season(lead=lead,experiment=experiment,
                                                      ACC_dictionary=ACC_season, 
                                                      obs_anom_climp=obs_anom_climp,
                                                      gef_bc_acc=gef_bc_crps, ecm_bc_acc=ecm_bc_crps,
                                                     obs_source=obs_source))

        
    return(ACC_dictionary,ACC_season)


In [19]:
# print(ACC_dictionary.keys())

def save_CRPS_tests(var, ACC_dictionary, region_name,obs_source,season):

    t1 = grab_ACC_from_dict(dict_ = ACC_dictionary, var = var)
    # print(acc.keys()) 

    ############################### SINGLE PREDICTION, NO BIAS CORRECTION ##################################################################
    # acc = grab_ACC_from_dict(dict_ = ACC_dictionary, var = var)
    # t1 = subset_delete(dict_ = acc, subset = 'bias_corrected')
    # print(t1.keys())

    #Save the average ACC values to a dictionary for later plotting
    # t_base = subset_keep(dict_ = t1, subset = 'baseline')
    # t_unet= subset_delete(dict_ = t1, subset = 'baseline')
    # print(t_base.keys())
    # print(t_unet.keys())
    
    file_path = f'Data/CRPS_dict_values/{region_name}/Wk_{lead}'
    os.system(f'mkdir -p {file_path}')
    if not season:
        file_save = f'{file_path}/CRPS_vals_{obs_source}.pkl'
    else:
        file_save = f'{file_path}/CRPS_vals_{obs_source}_season.pkl'
    
    
    with open(file_save, 'wb') as file:
        pickle.dump(t1, file)

    # plot_files_ACC(test_file = t1, var = var, name_of_test = f'{var} Single prediction ACC - No bias correction')

 
    return(0)



In [20]:
def grab_ACC_from_dict(dict_,var):
    crps = {key: value for key, value in dict_.items() if f'{var}_CRPS' in key}
    return(crps)

In [21]:
for lead in [1,2,3,4]:
    CRPS_dictionary,CRPS_season = run_CRPS(lead,region_name,obs_source)
    print(CRPS_dictionary)
    '''As a note, week 1 doesn't ever have any experimental runs for EX18-EX21, also EX26 is not a model that I ran'''
    save_CRPS_tests(var = 'RZSM', ACC_dictionary=CRPS_dictionary, region_name = region_name,obs_source=obs_source, season=False)
    save_CRPS_tests(var = 'RZSM', ACC_dictionary=CRPS_season, region_name = region_name,obs_source=obs_source, season=True)

stop

Working on lead 1
GEFS RZSM baseline value from reforecast (training, validation, testing) (no pre-processing other than anomaly computed.)
Maximum value in file is 0.2466011941432953
Minimum value in file is -0.9952380657196045
ECMWF RZSM baseline value from reforecast (training, validation, testing) (no pre-processing other than anomaly computed.)
Maximum value in file is 0.9951488375663757
Minimum value in file is -0.38824397325515747

Now loading Verification Data from Observations.

Loading obs_soilw_bgrnd_GLEAM_lead_1_train_masked.npy
Test prediction shape: (3, 1144, 48, 96, 1)
Shape of prediction RZSM: (1144, 48, 96, 1)
ECMWF RZSM anomaly prediction value from UNET
Maximum value in file is nan
Minimum value in file is nan
GEFSv12 RZSM anomaly prediction value from UNET
Maximum value in file is 0.15982161462306976
Minimum value in file is -0.2299593836069107


/glade/derecho/scratch/klesinger/tmp/ipykernel_115790/1585910870.py:3: RuntimeWarning: All-NaN axis encountered
  print(f'Maximum value in file is {np.nanmax(file[putils.xarray_varname(file)])}')
/glade/derecho/scratch/klesinger/tmp/ipykernel_115790/1585910870.py:4: RuntimeWarning: All-NaN axis encountered
  print(f'Minimum value in file is {np.nanmin(file[putils.xarray_varname(file)])}')
/glade/work/klesinger/conda-envs/tf212gpu_new/lib/python3.10/site-packages/climpred/classes.py:2357: UserWarning: rename 'time' to 'init' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  result = result.rename({"time": "init"})
/glade/work/klesinger/conda-envs/tf212gpu_new/lib/python3.10/site-packages/climpred/classes.py:2357: UserWarning: rename 'time' to 'init' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  result = result.rename({"time": "init

GEFS RZSM baseline anomaly from reforecast. No post-processing. 

Maximum value in file is 0.2466011941432953
Minimum value in file is -0.18918319046497345
ECMWF RZSM baseline anomaly from reforecast. No post-processing. 

Maximum value in file is 0.2832668125629425
Minimum value in file is -0.32933181524276733


/glade/work/klesinger/conda-envs/tf212gpu_new/lib/python3.10/site-packages/climpred/classes.py:2357: UserWarning: rename 'time' to 'init' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  result = result.rename({"time": "init"})
/glade/work/klesinger/conda-envs/tf212gpu_new/lib/python3.10/site-packages/climpred/classes.py:2357: UserWarning: rename 'time' to 'init' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  result = result.rename({"time": "init"})


GEFS RZSM baseline value from reforecast (training, validation, testing) (no pre-processing other than anomaly computed.)
Maximum value in file is 0.2466011941432953
Minimum value in file is -0.9952380657196045
ECMWF RZSM baseline value from reforecast (training, validation, testing) (no pre-processing other than anomaly computed.)
Maximum value in file is 0.9951488375663757
Minimum value in file is -0.38824397325515747

Now loading Verification Data from Observations.

Loading obs_soilw_bgrnd_GLEAM_lead_1_train_masked.npy
Test prediction shape: (3, 1144, 48, 96, 1)
Shape of prediction RZSM: (1144, 48, 96, 1)
ECMWF RZSM anomaly prediction value from UNET
Maximum value in file is nan
Minimum value in file is nan
GEFSv12 RZSM anomaly prediction value from UNET
Maximum value in file is 0.15982161462306976
Minimum value in file is -0.2299593836069107


...

/glade/work/klesinger/conda-envs/tf212gpu_new/lib/python3.10/site-packages/climpred/classes.py:2357: UserWarning: rename 'time' to 'init' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  result = result.rename({"time": "init"})
/glade/work/klesinger/conda-envs/tf212gpu_new/lib/python3.10/site-packages/climpred/classes.py:2357: UserWarning: rename 'time' to 'init' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  result = result.rename({"time": "init"})


GEFS RZSM baseline value from reforecast (training, validation, testing) (no pre-processing other than anomaly computed.)
Maximum value in file is 0.26432961225509644
Minimum value in file is -0.22833965718746185
ECMWF RZSM baseline value from reforecast (training, validation, testing) (no pre-processing other than anomaly computed.)
Maximum value in file is 0.9951485991477966
Minimum value in file is -0.9666666388511658

Now loading Verification Data from Observations.

Loading obs_soilw_bgrnd_GLEAM_lead_4_train_masked.npy
Test prediction shape: (3, 1144, 48, 96, 1)
Shape of prediction RZSM: (1144, 48, 96, 1)
ECMWF RZSM anomaly prediction value from UNET
Maximum value in file is 0.17487742006778717
Minimum value in file is -0.22833965718746185
GEFSv12 RZSM anomaly prediction value from UNET
Maximum value in file is 0.17235352098941803
Minimum value in file is -0.22833965718746185


/glade/work/klesinger/conda-envs/tf212gpu_new/lib/python3.10/site-packages/climpred/classes.py:2357: UserWarning: rename 'time' to 'init' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  result = result.rename({"time": "init"})
/glade/work/klesinger/conda-envs/tf212gpu_new/lib/python3.10/site-packages/climpred/classes.py:2357: UserWarning: rename 'time' to 'init' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  result = result.rename({"time": "init"})
/glade/work/klesinger/conda-envs/tf212gpu_new/lib/python3.10/site-packages/climpred/classes.py:2357: UserWarning: rename 'time' to 'init' does not create an index anymore. Try using swap_dims instead or use set_index after rename to create an indexed coordinate.
  result = result.rename({"time": "init"})
/glade/work/klesinger/conda-envs/tf212gpu_new/lib/python3.10/site-packages/climpre

{'Wk4_EX0_MEM_RZSM_CRPS': <xarray.Dataset> Size: 4MB
Dimensions:     (init: 104, lat: 48, lon: 96)
Coordinates:
  * lon         (lon) float64 768B 238.0 238.5 239.0 239.5 ... 284.5 285.0 285.5
  * lat         (lat) float64 384B 50.0 49.5 49.0 48.5 ... 28.0 27.5 27.0 26.5
    lead        float64 8B 4.0
    valid_time  (init) object 832B 2018-01-31 00:00:00 ... 2020-01-22 00:00:00
    init        (init) object 832B 2018-01-03 00:00:00 ... 2019-12-25 00:00:00
    skill       <U11 44B 'initialized'
Data variables:
    crps        (init, lat, lon) float64 4MB 0.005855 0.004878 ... nan nan
Attributes:
    Description:                   New data added to file
    lead:                          days
    prediction_skill_software:     climpred https://climpred.readthedocs.io/
    skill_calculated_by_function:  HindcastEnsemble.verify()
    number_of_members:             11
    alignment:                     same_inits
    metric:                        crps
    comparison:                    m2

NameError: name 'stop' is not defined